# 02 — Dislocation Analysis

Statistical analysis of extreme upside dislocations.

- Descriptive stats by taxonomy bucket
- Feature distributions (float, short interest, market cap)
- Continuation vs reversal analysis
- Clustering to validate taxonomy
- Logistic regression: what predicts 1000%+ given 500%+?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

API = "http://localhost:8000/quant/research"

def api(method, path, **kwargs):
    resp = getattr(requests, method)(f"{API}{path}", **kwargs)
    resp.raise_for_status()
    return resp.json()

Matplotlib is building the font cache; this may take a moment.
/Users/erichurchey/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# Load full dataset from API
events = api("get", "/events?limit=1000")
df = pd.DataFrame(events)
print(f"Loaded {len(df)} dislocation events")
df.head()

Loaded 92 dislocation events


,id,symbol,company_name,security_type,event_start_date,event_end_date,peak_date,price_start,price_peak,price_end_3d,return_1d_pct,return_3d_pct,return_peak_pct,label_a,label_b,day_after_continuation,volume_event_day,volume_avg_20d_pre,volume_ratio,shares_outstanding,float_shares,market_cap_pre,short_interest_shares,short_pct_float,days_to_cover,sector,industry,exchange,ipo_date,days_since_ipo,catalyst_summary,news_count_event_day,options_available,iv_pre_event,call_oi_pre,put_oi_pre,taxonomy_bucket,taxonomy_confidence,taxonomy_notes,data_source,created_at
0,8,BBIG,"Vinco Ventures, Inc.",common,2025-11-05T00:00:00,2025-11-10T00:00:00,2025-11-10T00:00:00,0.0005,0.0300,0.0198,1880.00,3860.00,5900.00,1,1,0.0,17611.0,4930.05,3.57,13000000.0,11665321.0,6500.0,2108418.0,0.1622,1.23,Consumer Cyclical,Leisure,PNK,None,NaN,"Tyde spinoff, short squeeze setup",None,0,None,None,None,short_squeeze,0.9,"Seed list: Tyde spinoff, short squeeze setup",yfinance,2026-03-26T20:27:30.270997
1,5,BBIG,"Vinco Ventures, Inc.",common,2024-12-31T00:00:00,2025-01-06T00:00:00,2025-01-06T00:00:00,0.0004,0.0120,0.0120,48.15,2900.00,2900.00,1,1,0.0,50525.0,21537.30,2.35,13000000.0,11665321.0,5200.0,2108418.0,0.1622,1.23,Consumer Cyclical,Leisure,PNK,None,NaN,"Tyde spinoff, short squeeze setup",None,0,None,None,None,short_squeeze,0.9,"Seed list: Tyde spinoff, short squeeze setup",yfinance,2026-03-26T20:27:30.270996
2,6,BBIG,"Vinco Ventures, Inc.",common,2025-01-15T00:00:00,2025-01-21T00:00:00,2025-01-21T00:00:00,0.0010,0.0300,0.0300,2900.00,2900.00,2900.00,1,1,0.0,4687.0,28892.70,0.16,13000000.0,11665321.0,13000.0,2108418.0,0.1622,1.23,Consumer Cyclical,Leisure,PNK,None,NaN,"Tyde spinoff, short squeeze setup",None,0,None,None,None,short_squeeze,0.9,"Seed list: Tyde spinoff, short squeeze setup",yfinance,2026-03-26T20:27:30.270996
3,7,BBIG,"Vinco Ventures, Inc.",common,2025-05-06T00:00:00,2025-05-09T00:00:00,2025-05-09T00:00:00,0.0004,0.0090,0.0090,2900.00,2150.00,2150.00,1,1,0.0,2382.0,1934.80,1.23,13000000.0,11665321.0,5200.0,2108418.0,0.1622,1.23,Consumer Cyclical,Leisure,PNK,None,NaN,"Tyde spinoff, short squeeze setup",None,0,None,None,None,short_squeeze,0.9,"Seed list: Tyde spinoff, short squeeze setup",yfinance,2026-03-26T20:27:30.270997
4,3,BBIG,"Vinco Ventures, Inc.",common,2024-07-03T00:00:00,2024-07-09T00:00:00,2024-07-08T00:00:00,0.0006,0.0101,0.0100,0.00,1566.67,1583.33,1,1,0.0,0.0,4258.45,0.00,13000000.0,11665321.0,7800.0,2108418.0,0.1622,1.23,Consumer Cyclical,Leisure,PNK,None,NaN,"Tyde spinoff, short squeeze setup",None,0,None,None,None,short_squeeze,0.9,"Seed list: Tyde spinoff, short squeeze setup",yfinance,2026-03-26T20:27:30.270995


## 1. Descriptive Statistics

In [3]:
# Summary stats for key numeric features
numeric_cols = [
    "return_3d_pct", "return_1d_pct", "return_peak_pct",
    "volume_ratio", "float_shares", "market_cap_pre",
    "short_pct_float", "days_to_cover", "days_since_ipo"
]
available = [c for c in numeric_cols if c in df.columns]
df[available].describe().round(2)

,return_3d_pct,return_1d_pct,return_peak_pct,volume_ratio,float_shares,market_cap_pre,short_pct_float,days_to_cover,days_since_ipo
count,92.00,92.00,92.00,92.00,9.200000e+01,9.200000e+01,92.00,92.00,3.00
mean,379.77,240.44,540.17,79.67,8.827056e+07,3.070014e+09,0.13,2.92,7320.00
std,625.01,513.02,870.02,250.60,1.293716e+08,1.641967e+10,0.09,3.41,206.57
min,-81.94,0.00,0.00,0.00,3.173944e+06,1.300000e+03,0.00,0.08,7096.00
25%,111.58,35.03,144.28,0.62,1.166532e+07,2.340000e+04,0.02,1.23,7228.50
50%,182.48,100.00,285.12,3.80,1.166532e+07,1.748421e+06,0.16,1.23,7361.00
75%,333.17,206.60,468.22,20.78,1.057062e+08,7.446616e+08,0.16,3.30,7432.00
max,3860.00,2900.00,5900.00,1433.30,5.246068e+08,1.545578e+11,0.31,12.79,7503.00


In [ ]:
# Label distribution
print("Label A (500%+):", df["label_a"].sum())
print("Label B (1000%+):", df["label_b"].sum())
print(f"\nContinuation rate (day after up):")
cont = df["day_after_continuation"].dropna()
if len(cont) > 0:
    print(f"  {cont.mean():.1%} ({int(cont.sum())}/{len(cont)} events)")

## 2. Taxonomy Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bucket counts
bucket_counts = df["taxonomy_bucket"].value_counts()
bucket_counts.plot.barh(ax=axes[0], color="steelblue")
axes[0].set_title("Events by Taxonomy Bucket")
axes[0].set_xlabel("Count")

# Average return by bucket
bucket_returns = df.groupby("taxonomy_bucket")["return_3d_pct"].mean().sort_values()
bucket_returns.plot.barh(ax=axes[1], color="coral")
axes[1].set_title("Mean 3-Day Return by Bucket")
axes[1].set_xlabel("Return %")

plt.tight_layout()
plt.show()

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 3-day return distribution
df["return_3d_pct"].hist(bins=30, ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("3-Day Return %")
axes[0, 0].axvline(x=1000, color="red", linestyle="--", label="1000% threshold")
axes[0, 0].legend()

# Volume ratio
vol = df["volume_ratio"].dropna()
if len(vol) > 0:
    vol.clip(upper=vol.quantile(0.95)).hist(bins=30, ax=axes[0, 1], color="coral")
axes[0, 1].set_title("Volume Ratio (event day / 20d avg)")

# Float shares (log scale)
flt = df["float_shares"].dropna()
if len(flt) > 0:
    np.log10(flt[flt > 0]).hist(bins=30, ax=axes[0, 2], color="green")
axes[0, 2].set_title("Float Shares (log10)")

# Market cap (log scale)
mcap = df["market_cap_pre"].dropna()
if len(mcap) > 0:
    np.log10(mcap[mcap > 0]).hist(bins=30, ax=axes[1, 0], color="purple")
axes[1, 0].set_title("Market Cap Pre-Event (log10)")

# Short % of float
si = df["short_pct_float"].dropna()
if len(si) > 0:
    si.hist(bins=30, ax=axes[1, 1], color="orange")
axes[1, 1].set_title("Short % of Float")

# Days since IPO
ipo = df["days_since_ipo"].dropna()
if len(ipo) > 0:
    ipo.clip(upper=ipo.quantile(0.95)).hist(bins=30, ax=axes[1, 2], color="teal")
axes[1, 2].set_title("Days Since IPO")

plt.tight_layout()
plt.show()

## 4. Continuation vs Reversal Analysis
Does the stock keep running on day 4, or does it reverse?

In [ ]:
cont_df = df[df["day_after_continuation"].notna()].copy()
if len(cont_df) > 0:
    cont_df["continued"] = cont_df["day_after_continuation"].astype(int)

    # Continuation rate by bucket
    bucket_cont = cont_df.groupby("taxonomy_bucket")["continued"].agg(["mean", "count"])
    bucket_cont.columns = ["continuation_rate", "n_events"]
    bucket_cont = bucket_cont.sort_values("continuation_rate", ascending=False)
    print("Continuation rate by bucket:")
    print(bucket_cont.round(3))

    # Continuation rate by label
    print(f"\nLabel A continuation: {cont_df[cont_df['label_a']==1]['continued'].mean():.1%}")
    label_b = cont_df[cont_df["label_b"]==1]
    if len(label_b) > 0:
        print(f"Label B continuation: {label_b['continued'].mean():.1%}")
else:
    print("No continuation data available yet. Run enrichment first.")

## 5. Cluster Analysis
Do the natural clusters match our taxonomy buckets?

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Select numeric features for clustering
cluster_cols = [
    "return_3d_pct", "volume_ratio", "float_shares",
    "market_cap_pre", "short_pct_float", "days_since_ipo"
]
available = [c for c in cluster_cols if c in df.columns]
cluster_df = df[available].dropna()

if len(cluster_df) >= 10:
    # Standardize
    scaler = StandardScaler()
    X = scaler.fit_transform(cluster_df)

    # K-means with k = number of non-trivial taxonomy buckets
    n_clusters = min(6, len(cluster_df) // 3)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(X)

    # PCA for visualization
    pca = PCA(n_components=2)
    X_2d = pca.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cluster assignments
    scatter = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=labels, cmap="tab10", alpha=0.7)
    axes[0].set_title(f"K-Means Clusters (k={n_clusters})")
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    plt.colorbar(scatter, ax=axes[0])

    # Taxonomy buckets for comparison
    taxonomy_labels = df.loc[cluster_df.index, "taxonomy_bucket"].fillna("unknown")
    unique_buckets = taxonomy_labels.unique()
    color_map = {b: i for i, b in enumerate(unique_buckets)}
    colors = [color_map[b] for b in taxonomy_labels]
    scatter2 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=colors, cmap="tab10", alpha=0.7)
    axes[1].set_title("Taxonomy Buckets")
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")

    plt.tight_layout()
    plt.show()

    print(f"\nExplained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
else:
    print(f"Only {len(cluster_df)} events with complete features — need more data for clustering.")

## 6. Logistic Regression: What predicts 1000%+ (Label B)?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

feature_cols = [
    "volume_ratio", "float_shares", "market_cap_pre",
    "short_pct_float", "days_since_ipo"
]
available = [c for c in feature_cols if c in df.columns]

reg_df = df[available + ["label_b"]].dropna()

if len(reg_df) >= 20 and reg_df["label_b"].sum() >= 3:
    X = reg_df[available]
    y = reg_df["label_b"]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    lr = LogisticRegression(random_state=42, max_iter=1000)
    scores = cross_val_score(lr, X_scaled, y, cv=min(5, len(reg_df) // 5), scoring="roc_auc")
    print(f"Cross-val AUC: {scores.mean():.3f} +/- {scores.std():.3f}")

    # Fit on all data for coefficients
    lr.fit(X_scaled, y)
    coef_df = pd.DataFrame({"feature": available, "coefficient": lr.coef_[0]})
    coef_df = coef_df.sort_values("coefficient", ascending=False)
    print("\nFeature importance (logistic regression coefficients):")
    print(coef_df.to_string(index=False))
else:
    print(f"Need more data: {len(reg_df)} events, {reg_df['label_b'].sum() if len(reg_df) > 0 else 0} Label B.")
    print("Run the full pipeline first to build a larger dataset.")

## 7. Summary Table for Paper

In [ ]:
# Summary table by taxonomy bucket
if "taxonomy_bucket" in df.columns and df["taxonomy_bucket"].notna().any():
    summary = df.groupby("taxonomy_bucket").agg(
        count=("symbol", "count"),
        mean_return_3d=("return_3d_pct", "mean"),
        median_return_3d=("return_3d_pct", "median"),
        mean_volume_ratio=("volume_ratio", "mean"),
        mean_float=("float_shares", "mean"),
        mean_market_cap=("market_cap_pre", "mean"),
        label_b_rate=("label_b", "mean"),
    ).round(2)
    summary = summary.sort_values("count", ascending=False)
    print("Summary by Taxonomy Bucket:")
    summary